In [1]:
from google.colab import files
uploaded = files.upload()

Saving RGB_Objects.zip to RGB_Objects.zip


In [2]:
import os
import zipfile

zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name, "r") as zip_file:
    zip_file.extractall("/content/dataset")

root = "/content/dataset/RGB_Objects"

for color in ["red", "green", "blue"]:
    folder = os.path.join(root, color)
    image_files = [
        name for name in os.listdir(folder)
        if name.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp"))
    ]
    print(color, ":", len(image_files), "photos")

red : 30 photos
green : 30 photos
blue : 30 photos


In [3]:
import os
import cv2
import numpy as np

X = []
y = []
image_types = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

for color in ["red", "green", "blue"]:
    folder = os.path.join(root, color)
    count_before = 0

    for filename in os.listdir(folder):
        if filename.lower().endswith(image_types):
            image = cv2.imread(os.path.join(folder, filename))

            if image is not None:
                image = cv2.resize(image, (64, 64))
                X.append(image.flatten() / 255.0)
                y.append(color)
                count_before += 1

    print(color, ":", count_before, "readable photos")

X = np.array(X)
y = np.array(y)

print("Total readable photos:", len(X))

red : 30 readable photos
green : 30 readable photos
blue : 30 readable photos
Total readable photos: 90


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = SVC(kernel="linear")
model.fit(X_train, y_train)

print("Training complete")
print("Training photos:", len(X_train))
print("Checking photos:", len(X_test))

Training complete
Training photos: 72
Checking photos: 18


In [5]:
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

labels = ["red", "green", "blue"]
matrix = confusion_matrix(y_test, y_pred, labels=labels)

print("\nConfusion matrix:")
print(pd.DataFrame(
    matrix,
    index=["Actual red", "Actual green", "Actual blue"],
    columns=["Predicted red", "Predicted green", "Predicted blue"]
))

Accuracy: 94.44%

Confusion matrix:
              Predicted red  Predicted green  Predicted blue
Actual red                5                1               0
Actual green              0                6               0
Actual blue               0                0               6


In [6]:
import joblib
from google.colab import files

joblib.dump(model, "rgb_object_classifier.pkl")
files.download("rgb_object_classifier.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>